<a href="https://www.kaggle.com/code/mrrogueknight/receiptiq?scriptVersionId=312700648" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [ ]:
import os

# Show only first 20 files
count = 0
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))
        count += 1
        if count >= 20:
            print("... more files hidden")
            break
    if count >= 20:
        break

# Cell 1: SROIE2019 Receipt Dataset Analysis

In [ ]:
# ==========================================
# Professional Kaggle Dataset Analyzer
# SROIE2019 Receipt Dataset
# ==========================================

import os
from pathlib import Path
import pandas as pd

DATASET = Path("/kaggle/input/datasets/urbikn/sroie-datasetv2/SROIE2019")

# ------------------------------------------
# Helper Functions
# ------------------------------------------

def count_files(folder):
    return sum(1 for f in folder.rglob("*") if f.is_file())

def folder_size_mb(folder):
    total = sum(f.stat().st_size for f in folder.rglob("*") if f.is_file())
    return round(total / (1024**2), 2)

def sample_files(folder, n=3):
    files = [str(f.relative_to(DATASET)) for f in folder.rglob("*") if f.is_file()]
    return ", ".join(files[:n])

# ------------------------------------------
# Summary Table
# ------------------------------------------

rows = []

for item in sorted(DATASET.iterdir()):
    if item.is_dir():
        rows.append({
            "Folder": item.name,
            "Total Files": count_files(item),
            "Size (MB)": folder_size_mb(item),
            "Sample Files": sample_files(item)
        })

summary_df = pd.DataFrame(rows)

print("=" * 70)
print("DATASET SUMMARY")
print("=" * 70)
display(summary_df)

# ------------------------------------------
# Detailed Structure
# ------------------------------------------

print("\n" + "=" * 70)
print("FOLDER STRUCTURE")
print("=" * 70)

for item in sorted(DATASET.iterdir()):
    if item.is_dir():
        print(f"\nFolder Name : {item.name}")
        print(f"Total Files : {count_files(item)}")
        print(f"Size (MB)   : {folder_size_mb(item)}")

        subfolders = [x.name for x in item.iterdir() if x.is_dir()]
        files = [x.name for x in item.iterdir() if x.is_file()]

        if subfolders:
            print("Subfolders  :", ", ".join(subfolders[:10]))

        if files:
            print("Files       :", ", ".join(files[:10]))

# ------------------------------------------
# SROIE Specific Validation
# ------------------------------------------

print("\n" + "=" * 70)
print("SROIE DATA VALIDATION")
print("=" * 70)

test_box = DATASET / "test" / "box"

if test_box.exists():
    txt_files = list(test_box.glob("*.txt"))
    print("Test Annotation Files :", len(txt_files))
    print("Sample Files          :", ", ".join([f.name for f in txt_files[:10]]))

model_dir = DATASET / "layoutlm-base-uncased"

if model_dir.exists():
    print("\nPretrained Model Directory Found")
    print("Contained Files:")

    for f in sorted(model_dir.iterdir()):
        print("-", f.name)

print("\nAnalysis Complete")

# EXPENSE TRACKER

In [ ]:
import gradio as gr
from PIL import Image
import easyocr
import cv2
import re
import numpy as np
from datetime import datetime
import hashlib
from collections import defaultdict, deque
from typing import Dict, List, Optional, Tuple
import pandas as pd
import os

# Suppress deprecation warnings
import warnings
warnings.filterwarnings('ignore')

# ============================================
# BACKEND LOGIC
# ============================================

class ReceiptManager:
    def __init__(self):
        self.receipts: List[Dict] = []
        self.receipt_index: set = set()
        self.category_totals: Dict[str, float] = defaultdict(float)
        self.store_totals: Dict[str, float] = defaultdict(float)
        self.monthly_totals: Dict[str, float] = defaultdict(float)
        self.recent_receipts: deque = deque(maxlen=20)
        self.total_spent: float = 0.0
        self.receipt_count: int = 0
        
    def add_receipt(self, receipt: Dict) -> bool:
        key = f"{receipt['store']}|{receipt['date']}|{receipt['amount']:.2f}"
        if key in self.receipt_index:
            return False
        self.receipts.append(receipt)
        self.receipt_index.add(key)
        self.receipt_count += 1
        self.total_spent += receipt['amount']
        self.category_totals[receipt['category']] += receipt['amount']
        self.store_totals[receipt['store']] += receipt['amount']
        month_key = receipt['date'][:7]
        self.monthly_totals[month_key] += receipt['amount']
        self.recent_receipts.appendleft(receipt)
        return True
    
    def clear_all(self):
        self.receipts.clear()
        self.receipt_index.clear()
        self.category_totals.clear()
        self.store_totals.clear()
        self.monthly_totals.clear()
        self.recent_receipts.clear()
        self.total_spent = 0.0
        self.receipt_count = 0
    
    def get_total(self) -> float:
        return self.total_spent
    
    def get_average(self) -> float:
        return self.total_spent / self.receipt_count if self.receipt_count > 0 else 0.0
    
    def get_monthly(self) -> float:
        current_month = datetime.now().strftime("%Y-%m")
        return self.monthly_totals.get(current_month, 0.0)
    
    def get_top_category(self) -> Tuple[str, float]:
        if not self.category_totals:
            return ("None", 0.0)
        return max(self.category_totals.items(), key=lambda x: x[1])
    
    def get_recent(self, limit: int = 5) -> List[Dict]:
        return list(self.recent_receipts)[:limit]
    
    def get_category_breakdown(self) -> Dict[str, float]:
        return dict(self.category_totals)

class OCRProcessor:
    _instance = None
    _reader = None
    
    def __new__(cls):
        if cls._instance is None:
            cls._instance = super().__new__(cls)
        return cls._instance
    
    @classmethod
    def get_reader(cls):
        if cls._reader is None:
            try:
                cls._reader = easyocr.Reader(['en'], gpu=False, verbose=False)
            except Exception:
                cls._reader = None
        return cls._reader
    
    def process(self, image: np.ndarray) -> Tuple[Optional[Dict], Optional[str]]:
        reader = self.get_reader()
        if reader is None:
            return None, "OCR engine unavailable"
        
        try:
            if len(image.shape) == 3:
                gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
            else:
                gray = image
            
            results = reader.readtext(gray, paragraph=False)
            if not results:
                return None, "No text detected"
            
            text_lines = [r[1] for r in results]
            full_text = " ".join(text_lines).lower()
            
            store = "Unknown Store"
            for line in text_lines[:5]:
                if len(line) > 3 and not any(c.isdigit() for c in line[:2]) and len(line) < 40:
                    store = line.strip()
                    break
            
            amounts = re.findall(r'(\d+\.\d{2})', full_text)
            total = max([float(a) for a in amounts]) if amounts else 0.0
            
            if total == 0:
                return None, "No amount found"
            
            date_match = re.search(r'(\d{1,2}[/-]\d{1,2}[/-]\d{2,4})', full_text)
            date = date_match.group(1) if date_match else datetime.now().strftime("%Y-%m-%d")
            
            categories = {
                "Groceries": ["mart", "grocery", "market", "walmart", "kroger", "aldi", "whole foods"],
                "Food & Dining": ["restaurant", "cafe", "food", "dining", "starbucks", "mcdonald", "pizza"],
                "Shopping": ["mall", "shop", "amazon", "target", "best buy"],
                "Transport": ["uber", "lyft", "taxi", "bus", "train", "petrol", "gas"],
                "Bills": ["electricity", "water", "internet", "phone", "netflix"],
                "Healthcare": ["pharmacy", "clinic", "hospital", "doctor", "medicine"]
            }
            
            category = "Shopping"
            for cat, keywords in categories.items():
                if any(kw in full_text for kw in keywords):
                    category = cat
                    break
            
            return {
                "id": hashlib.md5(f"{store}{date}{total}".encode()).hexdigest()[:8],
                "store": store.title(),
                "date": date,
                "amount": total,
                "category": category,
                "timestamp": datetime.now().isoformat()
            }, None
        except Exception as e:
            return None, str(e)

receipt_manager = ReceiptManager()
ocr_processor = OCRProcessor()

# ============================================
# CHAT RESPONSES
# ============================================

def get_summary():
    if receipt_manager.receipt_count == 0:
        return "No receipts yet. Upload your first receipt."
    top_cat, top_amt = receipt_manager.get_top_category()
    return f"""**Expense Summary**

Total Spent: **${receipt_manager.get_total():,.2f}**
Receipts: **{receipt_manager.receipt_count}**
Average: **${receipt_manager.get_average():,.2f}**
Top Category: **{top_cat}** (${top_amt:,.2f})"""

def get_recent():
    recent = receipt_manager.get_recent(5)
    if not recent:
        return "No receipts yet."
    return "**Recent Receipts**\n\n" + "\n".join([f"• **{r['store']}** — ${r['amount']:.2f} ({r['date']})" for r in recent])

def get_categories():
    cats = receipt_manager.get_category_breakdown()
    if not cats:
        return "No categories yet."
    total = receipt_manager.get_total()
    return "**Category Breakdown**\n\n" + "\n".join([f"• **{cat}:** ${amt:,.2f} ({amt/total*100:.0f}%)" for cat, amt in sorted(cats.items(), key=lambda x: x[1], reverse=True)])

def get_monthly():
    if receipt_manager.receipt_count == 0:
        return "No receipts yet."
    monthly = receipt_manager.get_monthly()
    return f"**This Month**\n\n${monthly:,.2f}"

def get_advice():
    if receipt_manager.receipt_count == 0:
        return "Upload receipts to get advice."
    top_cat, _ = receipt_manager.get_top_category()
    tips = {
        "Groceries": "Plan meals and use a shopping list to avoid impulse purchases.",
        "Food & Dining": "Cook at home 2 more days per week to save significantly.",
        "Shopping": "Wait 24 hours before non-essential purchases.",
        "Transport": "Consider public transit or carpooling to reduce costs.",
        "Bills": "Review and cancel unused subscriptions monthly.",
        "Healthcare": "Compare prices at different pharmacies for medications."
    }
    return f"**Savings Tip**\n\n{tips.get(top_cat, 'Set monthly budgets for each category to track spending effectively.')}"

# ============================================
# MULTI-RECEIPT UPLOAD
# ============================================

def process_multiple_uploads(files, history):
    """Process multiple receipts at once"""
    if not files:
        return history, ""
    
    processed_count = 0
    duplicate_count = 0
    failed_count = 0
    total_amount = 0.0
    receipts_data = []
    
    for file in files:
        try:
            image = Image.open(file.name)
            img_array = np.array(image)
            receipt_data, error = ocr_processor.process(img_array)
            
            if error or receipt_data is None:
                failed_count += 1
                continue
            
            if receipt_manager.add_receipt(receipt_data):
                processed_count += 1
                total_amount += receipt_data['amount']
                receipts_data.append(receipt_data)
            else:
                duplicate_count += 1
        except Exception:
            failed_count += 1
    
    # User message
    file_msg = f"📎 Uploaded {len(files)} receipt(s)"
    history.append({"role": "user", "content": file_msg})
    
    # Assistant response
    if processed_count > 0:
        response = f"""**Processed {processed_count} receipts**

**Total detected:** ${total_amount:,.2f}"""
        if duplicate_count > 0:
            response += f"\n**{duplicate_count} duplicate(s) skipped**"
        if failed_count > 0:
            response += f"\n**{failed_count} failed to process**"
        
        from collections import Counter
        categories = [r['category'] for r in receipts_data]
        if categories:
            top_cat = Counter(categories).most_common(1)[0][0]
            response += f"\n\n**Top category:** {top_cat}"
        
        response += f"\n\n**Total spent:** ${receipt_manager.get_total():,.2f}\n**Receipts count:** {receipt_manager.receipt_count}"
    else:
        response = "❌ No receipts could be processed. Please ensure images are clear."
    
    history.append({"role": "assistant", "content": response})
    return history, ""

def send_message(message, history):
    if not message or not message.strip():
        return history, ""
    
    msg = message.lower()
    history.append({"role": "user", "content": message})
    
    if receipt_manager.receipt_count == 0:
        response = "No receipts yet. Upload receipts to get started."
    elif "summary" in msg or "total" in msg:
        response = get_summary()
    elif "recent" in msg:
        response = get_recent()
    elif "category" in msg:
        response = get_categories()
    elif "month" in msg:
        response = get_monthly()
    elif "advice" in msg or "tip" in msg:
        response = get_advice()
    else:
        response = f"""I can help with your {receipt_manager.receipt_count} receipts (${receipt_manager.get_total():.2f} total).

**Try these commands:**
• **summary** - Complete expense report
• **recent** - Last 5 receipts
• **categories** - Spending breakdown
• **this month** - Monthly total
• **advice** - Savings tips"""
    
    history.append({"role": "assistant", "content": response})
    return history, ""

def clear_all():
    receipt_manager.clear_all()
    return [], ""

def export_data():
    if receipt_manager.receipt_count == 0:
        return None
    df = pd.DataFrame(receipt_manager.receipts)
    filename = f"receipts_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
    df.to_csv(filename, index=False)
    return filename

def update_sidebar():
    """Update the expense tracker sidebar"""
    total = receipt_manager.get_total()
    count = receipt_manager.receipt_count
    avg = receipt_manager.get_average()
    monthly = receipt_manager.get_monthly()
    top_cat, top_amt = receipt_manager.get_top_category()
    
    # Category breakdown
    cats = receipt_manager.get_category_breakdown()
    category_bars = ""
    for cat, amt in sorted(cats.items(), key=lambda x: x[1], reverse=True)[:5]:
        percent = (amt / total) * 100 if total > 0 else 0
        category_bars += f"""
        <div style="margin-bottom: 12px;">
            <div style="display: flex; justify-content: space-between; font-size: 12px; margin-bottom: 4px;">
                <span style="color: #D1D5DB;">{cat}</span>
                <span style="color: #10A37F;">${amt:,.0f}</span>
            </div>
            <div style="background: #1F2937; border-radius: 8px; overflow: hidden;">
                <div style="background: #10A37F; width: {percent}%; height: 4px; border-radius: 8px;"></div>
            </div>
        </div>
        """
    
    if not category_bars:
        category_bars = '<div style="color: #6B7280; text-align: center; padding: 20px;">No data yet</div>'
    
    # Recent receipts
    recent = receipt_manager.get_recent(3)
    recent_html = ""
    for r in recent:
        recent_html += f"""
        <div style="padding: 8px 0; border-bottom: 1px solid rgba(255,255,255,0.05);">
            <div style="font-size: 13px; font-weight: 500; color: #F3F4F6;">{r['store'][:25]}</div>
            <div style="display: flex; justify-content: space-between; margin-top: 4px;">
                <span style="font-size: 11px; color: #9CA3AF;">{r['date']}</span>
                <span style="font-size: 12px; color: #10A37F;">${r['amount']:.2f}</span>
            </div>
        </div>
        """
    
    if not recent_html:
        recent_html = '<div style="color: #6B7280; text-align: center; padding: 20px;">No receipts yet</div>'
    
    return f"""
    <div style="padding: 20px; height: 100%; overflow-y: auto;">
        <div style="margin-bottom: 24px;">
            <div style="font-size: 20px; font-weight: 700; background: linear-gradient(135deg, #10A37F, #3B82F6); -webkit-background-clip: text; -webkit-text-fill-color: transparent;">ReceiptIQ</div>
            <div style="font-size: 11px; color: #6B7280; margin-top: 4px;">AI expense tracker</div>
        </div>
        
        <div style="background: #1A1F2E; border-radius: 16px; padding: 16px; margin-bottom: 20px;">
            <div style="display: grid; grid-template-columns: repeat(2, 1fr); gap: 16px;">
                <div>
                    <div style="font-size: 11px; color: #9CA3AF;">Total Spent</div>
                    <div style="font-size: 24px; font-weight: 700; color: #10A37F;">${total:,.2f}</div>
                </div>
                <div>
                    <div style="font-size: 11px; color: #9CA3AF;">Receipts</div>
                    <div style="font-size: 24px; font-weight: 700;">{count}</div>
                </div>
                <div>
                    <div style="font-size: 11px; color: #9CA3AF;">Average</div>
                    <div style="font-size: 20px; font-weight: 600;">${avg:,.0f}</div>
                </div>
                <div>
                    <div style="font-size: 11px; color: #9CA3AF;">This Month</div>
                    <div style="font-size: 20px; font-weight: 600;">${monthly:,.0f}</div>
                </div>
            </div>
        </div>
        
        <div style="margin-bottom: 20px;">
            <div style="font-size: 13px; font-weight: 600; margin-bottom: 12px; color: #E5E7EB;">Category Breakdown</div>
            {category_bars}
        </div>
        
        <div style="margin-bottom: 20px;">
            <div style="font-size: 13px; font-weight: 600; margin-bottom: 12px; color: #E5E7EB;">Top Category</div>
            <div style="background: #1A1F2E; border-radius: 12px; padding: 12px;">
                <div style="font-size: 16px; font-weight: 600; color: #10A37F;">{top_cat}</div>
                <div style="font-size: 12px; color: #9CA3AF;">${top_amt:,.2f} total</div>
            </div>
        </div>
        
        <div style="margin-bottom: 20px;">
            <div style="font-size: 13px; font-weight: 600; margin-bottom: 12px; color: #E5E7EB;">Recent Receipts</div>
            <div style="background: #1A1F2E; border-radius: 12px; padding: 12px;">
                {recent_html}
            </div>
        </div>
        
        <div style="display: flex; gap: 8px; margin-top: 20px;">
            <button onclick="document.querySelector('#clear-all-btn').click();" style="flex: 1; background: rgba(239,68,68,0.2); border: 1px solid rgba(239,68,68,0.3); border-radius: 40px; padding: 8px; color: #EF4444; cursor: pointer; font-size: 12px;">Clear All</button>
            <button onclick="document.querySelector('#export-btn').click();" style="flex: 1; background: rgba(16,163,127,0.2); border: 1px solid rgba(16,163,127,0.3); border-radius: 40px; padding: 8px; color: #10A37F; cursor: pointer; font-size: 12px;">Export CSV</button>
        </div>
    </div>
    """

# ============================================
# CHATGPT DESKTOP UI CSS
# ============================================

css = """
* {
    margin: 0;
    padding: 0;
    box-sizing: border-box;
}

body, .gradio-container {
    background: #0A0A0A !important;
    font-family: -apple-system, BlinkMacSystemFont, "SF Pro Text", "Inter", system-ui, sans-serif !important;
}

.gradio-container {
    max-width: 100% !important;
    margin: 0 !important;
    padding: 0 !important;
}

/* Hide Gradio footer */
footer, .gradio-container > footer {
    display: none !important;
}

/* Main layout - 20% sidebar, 80% chat */
.main-layout {
    display: flex !important;
    height: 100vh !important;
    width: 100% !important;
    gap: 0 !important;
}

/* Expense Tracker Sidebar - 20% */
.expense-sidebar {
    width: 20% !important;
    min-width: 260px !important;
    background: #0F141F !important;
    border-right: 1px solid rgba(255,255,255,0.06) !important;
    overflow-y: auto !important;
    height: 100vh !important;
    position: fixed !important;
    left: 0 !important;
    top: 0 !important;
}

/* Chat Area - 80% */
.chat-area {
    width: 80% !important;
    margin-left: 20% !important;
    display: flex !important;
    flex-direction: column !important;
    height: 100vh !important;
    background: #0A0A0A !important;
}

/* Center the conversation */
.chat-center {
    max-width: 800px !important;
    margin: 0 auto !important;
    width: 100% !important;
    height: 100% !important;
    display: flex !important;
    flex-direction: column !important;
}

/* Header - minimal */
.chat-header {
    padding: 20px 32px 8px 32px !important;
    border-bottom: none !important;
    background: transparent !important;
    flex-shrink: 0 !important;
}

.chat-header h1 {
    font-size: 20px !important;
    font-weight: 600 !important;
    background: linear-gradient(135deg, #10A37F, #3B82F6) !important;
    -webkit-background-clip: text !important;
    -webkit-text-fill-color: transparent !important;
    margin: 0 !important;
}

.chat-header p {
    font-size: 13px !important;
    color: #6B7280 !important;
    margin-top: 4px !important;
}

/* Chatbot area - scrollable conversation */
.chatbot-area {
    flex: 1 !important;
    overflow-y: auto !important;
    padding: 0 32px !important;
}

/* Floating composer - bottom center */
.composer-wrapper {
    position: sticky !important;
    bottom: 0 !important;
    background: linear-gradient(to top, #0A0A0A 80%, transparent) !important;
    padding: 20px 32px 32px 32px !important;
    flex-shrink: 0 !important;
}

.composer {
    background: #1A1A1A !important;
    border-radius: 32px !important;
    border: 1px solid #2D2D2D !important;
    padding: 8px 12px !important;
    display: flex !important;
    gap: 8px !important;
    align-items: center !important;
}

.composer textarea {
    background: transparent !important;
    border: none !important;
    color: white !important;
    font-size: 14px !important;
    padding: 8px 12px !important;
    resize: none !important;
    font-family: inherit !important;
}

.composer textarea:focus {
    outline: none !important;
    box-shadow: none !important;
}

.upload-btn {
    background: transparent !important;
    border: none !important;
    color: #9CA3AF !important;
    font-size: 20px !important;
    cursor: pointer !important;
    padding: 8px !important;
    min-width: 40px !important;
}

.upload-btn:hover {
    color: #10A37F !important;
}

.send-btn {
    background: transparent !important;
    border: none !important;
    color: #10A37F !important;
    cursor: pointer !important;
    padding: 8px 16px !important;
    font-weight: 500 !important;
}

.send-btn:hover {
    color: #0E8C6D !important;
}

/* Quick action chips */
.chips-row {
    display: flex !important;
    gap: 8px !important;
    flex-wrap: wrap !important;
    justify-content: center !important;
    margin: 0 32px 12px 32px !important;
    flex-shrink: 0 !important;
}

.chip {
    background: #1A1A1A !important;
    border: 1px solid #2D2D2D !important;
    border-radius: 100px !important;
    padding: 6px 14px !important;
    font-size: 12px !important;
    color: #9CA3AF !important;
    cursor: pointer !important;
    transition: all 0.2s !important;
}

.chip:hover {
    background: #10A37F !important;
    color: white !important;
    border-color: #10A37F !important;
}

/* Scrollbar */
::-webkit-scrollbar {
    width: 4px;
}

::-webkit-scrollbar-track {
    background: transparent;
}

::-webkit-scrollbar-thumb {
    background: #2D2D2D;
    border-radius: 4px;
}

::-webkit-scrollbar-thumb:hover {
    background: #10A37F;
}

/* Responsive */
@media (max-width: 768px) {
    .expense-sidebar {
        display: none !important;
    }
    .chat-area {
        width: 100% !important;
        margin-left: 0 !important;
    }
}
"""

# ============================================
# MAIN APP - 20% EXPENSE TRACKER + 80% CHAT
# ============================================

# Create the app without CSS parameter to avoid warning
with gr.Blocks() as demo:
    
    # Inject CSS via HTML
    gr.HTML(f"<style>{css}</style>")
    
    with gr.Row(elem_classes="main-layout"):
        
        # LEFT SIDEBAR - Expense Tracker (20%)
        with gr.Column(elem_classes="expense-sidebar"):
            sidebar_html = gr.HTML(update_sidebar())
        
        # RIGHT CHAT AREA (80%)
        with gr.Column(elem_classes="chat-area"):
            
            with gr.Column(elem_classes="chat-center"):
                
                # Header
                with gr.Row(elem_classes="chat-header"):
                    gr.HTML("""
                        <h1>Receipt Assistant</h1>
                        <p>Upload receipts and ask questions about your spending</p>
                    """)
                
                # Chatbot area
                chatbot = gr.Chatbot(
                    type="messages",
                    show_label=False,
                    elem_classes="chatbot-area",
                    height=500
                )
                
                # Quick action chips
                with gr.Row(elem_classes="chips-row"):
                    summary_chip = gr.Button("Summary", elem_classes="chip", size="sm")
                    recent_chip = gr.Button("Recent", elem_classes="chip", size="sm")
                    categories_chip = gr.Button("Categories", elem_classes="chip", size="sm")
                    monthly_chip = gr.Button("This Month", elem_classes="chip", size="sm")
                    advice_chip = gr.Button("Advice", elem_classes="chip", size="sm")
                
                # Floating composer
                with gr.Row(elem_classes="composer-wrapper"):
                    with gr.Row(elem_classes="composer"):
                        upload_btn = gr.UploadButton(
                            "📎",
                            file_types=["image"],
                            file_count="multiple",
                            elem_classes="upload-btn",
                            scale=0
                        )
                        msg = gr.Textbox(
                            placeholder="Ask about your receipts...",
                            show_label=False,
                            scale=10,
                            lines=1
                        )
                        send_btn = gr.Button("Send", elem_classes="send-btn", scale=0)
    
    # Hidden buttons
    clear_btn = gr.Button("Clear", visible=False, elem_id="clear-all-btn")
    export_btn = gr.Button("Export", visible=False, elem_id="export-btn")
    
    # ========== EVENT HANDLERS ==========
    
    # Function to refresh sidebar
    def refresh_sidebar():
        return update_sidebar()
    
    # Multi-upload
    upload_btn.upload(process_multiple_uploads, [upload_btn, chatbot], [chatbot, msg]).then(
        refresh_sidebar, None, sidebar_html
    )
    
    # Send message
    send_btn.click(send_message, [msg, chatbot], [chatbot, msg]).then(
        lambda: "", None, msg
    ).then(
        refresh_sidebar, None, sidebar_html
    )
    
    msg.submit(send_message, [msg, chatbot], [chatbot, msg]).then(
        lambda: "", None, msg
    ).then(
        refresh_sidebar, None, sidebar_html
    )
    
    # Chip handlers
    summary_chip.click(lambda: send_message("summary", []), None, [chatbot, msg]).then(
        refresh_sidebar, None, sidebar_html
    )
    recent_chip.click(lambda: send_message("recent", []), None, [chatbot, msg]).then(
        refresh_sidebar, None, sidebar_html
    )
    categories_chip.click(lambda: send_message("categories", []), None, [chatbot, msg]).then(
        refresh_sidebar, None, sidebar_html
    )
    monthly_chip.click(lambda: send_message("this month", []), None, [chatbot, msg]).then(
        refresh_sidebar, None, sidebar_html
    )
    advice_chip.click(lambda: send_message("advice", []), None, [chatbot, msg]).then(
        refresh_sidebar, None, sidebar_html
    )
    
    # Clear and export
    clear_btn.click(clear_all, None, [chatbot, msg]).then(
        refresh_sidebar, None, sidebar_html
    )
    export_btn.click(export_data, None, None)
    
    # Initial sidebar load
    demo.load(refresh_sidebar, None, sidebar_html)

if __name__ == "__main__":
    demo.launch(share=True)

# AI Concepts Implemented in Receipt Expense Tracker

In [ ]:
# ===============================================================
# AI Concepts Implemented in Receipt Expense Tracker
# ===============================================================

# -----------------------------
# 1. Import Libraries
# -----------------------------
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)
from sklearn.linear_model import LinearRegression

# ===============================================================
# 2. Sample Receipt Dataset
# Simulates OCR text extracted from receipts
# ===============================================================
data = {
    "receipt_text": [
        "big bazaar grocery rice milk bread total 850",
        "dominos pizza cheese meal total 620",
        "uber trip airport fare total 430",
        "apollo pharmacy medicine tablet total 760",
        "amazon shopping headphones total 2499",
        "starbucks coffee latte total 320",
        "dmart grocery fruits vegetables total 910",
        "ola cab city ride total 290",
        "flipkart shopping mobile cover total 399",
        "mcdonald burger fries total 510"
    ],
    
    "category": [
        "Groceries",
        "Food",
        "Transport",
        "Healthcare",
        "Shopping",
        "Food",
        "Groceries",
        "Transport",
        "Shopping",
        "Food"
    ],
    
    "amount": [850,620,430,760,2499,320,910,290,399,510]
}

df = pd.DataFrame(data)

print("="*65)
print("Receipt Expense Tracker - AI Implementation Assignment")
print("="*65)

print("\nDataset Preview:\n")
print(df)

# ===============================================================
# 3. AI Concept: Text Search / Rule Based Intelligence
# Search Total Amount from Receipt Text
# ===============================================================
print("\n1. Rule Based Text Search")

def extract_total(text):
    match = re.search(r'total\s+(\d+)', text)
    return int(match.group(1)) if match else 0

df["detected_total"] = df["receipt_text"].apply(extract_total)

print(df[["receipt_text","detected_total"]].head())

# ===============================================================
# 4. AI Concept: Feature Extraction using TF-IDF
# ===============================================================
print("\n2. Feature Extraction using TF-IDF")

vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(df["receipt_text"])
y = df["category"]

print("Feature Matrix Shape:", X.shape)

# ===============================================================
# 5. AI Concept: Train-Test Split
# ===============================================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42
)

# ===============================================================
# 6. AI Concept: Naive Bayes Classification
# Predict Expense Category
# ===============================================================
print("\n3. Naive Bayes Expense Classification")

nb = MultinomialNB()
nb.fit(X_train, y_train)

pred_nb = nb.predict(X_test)

print("Predictions:", pred_nb)

# ===============================================================
# 7. Evaluation Metrics
# ===============================================================
print("\n4. Model Evaluation")

acc = accuracy_score(y_test, pred_nb)

print("Accuracy :", round(acc,2))

# ===============================================================
# 8. AI Concept: Support Vector Machine
# ===============================================================
print("\n5. Support Vector Machine")

svm = SVC(kernel="linear")
svm.fit(X_train, y_train)

pred_svm = svm.predict(X_test)

acc_svm = accuracy_score(y_test, pred_svm)

print("SVM Accuracy:", round(acc_svm,2))

# ===============================================================
# 9. AI Concept: Neural Network
# ===============================================================
print("\n6. Neural Network")

ann = MLPClassifier(hidden_layer_sizes=(16,), max_iter=2000, random_state=1)
ann.fit(X_train, y_train)

pred_ann = ann.predict(X_test)

acc_ann = accuracy_score(y_test, pred_ann)

print("ANN Accuracy:", round(acc_ann,2))

# ===============================================================
# 10. Compare Models
# ===============================================================
print("\n7. Model Comparison")

models = ["Naive Bayes","SVM","ANN"]
scores = [acc, acc_svm, acc_ann]

comparison = pd.DataFrame({
    "Model": models,
    "Accuracy": scores
})

print(comparison)

plt.figure(figsize=(8,4))
plt.bar(models, scores)
plt.title("Expense Category Prediction Accuracy")
plt.ylabel("Accuracy")
plt.ylim(0,1)
plt.show()

# ===============================================================
# 11. Regression for Monthly Expense Prediction
# ===============================================================
print("\n8. Linear Regression for Expense Forecasting")

months = np.array([1,2,3,4,5]).reshape(-1,1)
expenses = np.array([5200,6100,6900,7200,8100])

reg = LinearRegression()
reg.fit(months, expenses)

next_month = reg.predict([[6]])[0]

print("Predicted Expense for Month 6:", round(next_month,2))

# ===============================================================
# 12. Smart Category Detection for New Receipt
# ===============================================================
print("\n9. Real Time Prediction Example")

new_receipt = ["swiggy food order burger total 450"]

new_vector = vectorizer.transform(new_receipt)

best_model = svm
prediction = best_model.predict(new_vector)[0]

print("Receipt Text :", new_receipt[0])
print("Predicted Category :", prediction)

# ===============================================================
# 13. Dashboard Analytics
# ===============================================================
print("\n10. Expense Analytics")

summary = df.groupby("category")["amount"].sum().sort_values(ascending=False)
print(summary)

summary.plot(kind="bar", figsize=(8,4), title="Category Wise Total Spending")
plt.ylabel("Amount")
plt.show()

# ===============================================================
# 14. Final Pipeline Explanation
# ===============================================================
print("\nEnd-to-End AI Pipeline")
print("Receipt Image -> OCR Text -> Text Cleaning -> TFIDF Features")
print("-> AI Model -> Category Prediction -> Dashboard -> Export")

# AI Based Receipt Expense Tracker

In [ ]:
# ===============================================================
# AI Based Receipt Expense Tracker
# ===============================================================

# -----------------------------
# 1. Import Libraries
# -----------------------------
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score
from sklearn.linear_model import LinearRegression

plt.style.use("ggplot")

# ===============================================================
# 2. Create Multi-Person Receipt Dataset
# ===============================================================
data = {
    "person": [
        "Aman","Aman","Aman",
        "Riya","Riya","Riya",
        "Kunal","Kunal","Kunal",
        "Sneha","Sneha","Sneha"
    ],
    
    "receipt_text": [
        "big bazaar grocery rice milk total 850",
        "uber trip office total 430",
        "dominos pizza meal total 620",
        
        "amazon shopping shoes total 2499",
        "starbucks coffee total 320",
        "ola cab city ride total 290",
        
        "apollo pharmacy medicine total 760",
        "dmart grocery fruits total 910",
        "flipkart shopping headphones total 399",
        
        "mcdonald burger fries total 510",
        "uber airport trip total 600",
        "amazon bag shopping total 1200"
    ],
    
    "category": [
        "Groceries","Transport","Food",
        "Shopping","Food","Transport",
        "Healthcare","Groceries","Shopping",
        "Food","Transport","Shopping"
    ],
    
    "amount":[850,430,620,2499,320,290,760,910,399,510,600,1200]
}

df = pd.DataFrame(data)

print("="*70)
print("Receipt Expense Tracker - Advanced AI Analytics")
print("="*70)

print("\nDataset Preview:\n")
print(df)

# ===============================================================
# 3. Rule Based Amount Extraction
# ===============================================================
def extract_total(text):
    match = re.search(r'total\s+(\d+)', text)
    return int(match.group(1)) if match else 0

df["detected_total"] = df["receipt_text"].apply(extract_total)

# ===============================================================
# 4. Feature Extraction + Classification
# ===============================================================
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(df["receipt_text"])
y = df["category"]

X_train,X_test,y_train,y_test = train_test_split(
    X,y,test_size=0.30,random_state=42
)

# Naive Bayes
nb = MultinomialNB()
nb.fit(X_train,y_train)
pred_nb = nb.predict(X_test)
acc_nb = accuracy_score(y_test,pred_nb)

# SVM
svm = SVC(kernel="linear")
svm.fit(X_train,y_train)
pred_svm = svm.predict(X_test)
acc_svm = accuracy_score(y_test,pred_svm)

# ANN
ann = MLPClassifier(hidden_layer_sizes=(16,),max_iter=3000,random_state=1)
ann.fit(X_train,y_train)
pred_ann = ann.predict(X_test)
acc_ann = accuracy_score(y_test,pred_ann)

# ===============================================================
# 5. Model Accuracy Comparison
# ===============================================================
models = ["Naive Bayes","SVM","ANN"]
scores = [acc_nb,acc_svm,acc_ann]

plt.figure(figsize=(8,4))
plt.bar(models,scores)
plt.title("AI Model Accuracy Comparison")
plt.ylabel("Accuracy")
plt.ylim(0,1)
plt.show()

# ===============================================================
# 6. Person Wise Expense Analysis
# ===============================================================
print("\nPerson Wise Total Spending:\n")

person_total = df.groupby("person")["amount"].sum().sort_values(ascending=False)
print(person_total)

plt.figure(figsize=(8,4))
person_total.plot(kind="bar")
plt.title("Total Expense by Person")
plt.ylabel("Amount")
plt.show()

# ===============================================================
# 7. Category Wise Expense
# ===============================================================
cat_total = df.groupby("category")["amount"].sum()

plt.figure(figsize=(8,4))
cat_total.plot(kind="pie",autopct="%1.1f%%")
plt.title("Expense Category Distribution")
plt.ylabel("")
plt.show()

# ===============================================================
# 8. Spending Behaviour Analysis
# ===============================================================
print("\nSpending Statistics Per Person:\n")

stats = df.groupby("person")["amount"].agg(
    Total="sum",
    Mean="mean",
    Median="median",
    Max="max",
    Min="min",
    Transactions="count"
)

print(stats)

# ===============================================================
# 9. Detect Who Spends Most on What
# ===============================================================
print("\nFavourite Spending Category Per Person:\n")

fav = df.groupby(["person","category"])["amount"].sum().reset_index()
fav = fav.loc[fav.groupby("person")["amount"].idxmax()]
print(fav[["person","category","amount"]])

# ===============================================================
# 10. Monthly Forecast Example
# ===============================================================
months = np.array([1,2,3,4,5]).reshape(-1,1)
expenses = np.array([5000,6200,7100,7600,8300])

reg = LinearRegression()
reg.fit(months,expenses)

future = reg.predict([[6]])[0]

print("\nPredicted Expense for Next Month:", round(future,2))

plt.figure(figsize=(8,4))
plt.scatter(months,expenses,label="Past Expense")
plt.plot(months,reg.predict(months),label="Trend Line")
plt.scatter(6,future,s=100,label="Prediction")
plt.title("Monthly Expense Forecast")
plt.xlabel("Month")
plt.ylabel("Expense")
plt.legend()
plt.show()

# ===============================================================
# 11. Smart New Receipt Prediction
# ===============================================================
new_receipt = ["swiggy burger food total 450"]
new_vec = vectorizer.transform(new_receipt)

prediction = svm.predict(new_vec)[0]

print("\nNew Receipt Prediction")
print("Receipt:", new_receipt[0])
print("Predicted Category:", prediction)

# ===============================================================
# 12. Heatmap Style Pivot Analysis
# ===============================================================
pivot = pd.pivot_table(
    df,
    values="amount",
    index="person",
    columns="category",
    aggfunc="sum",
    fill_value=0
)

print("\nExpense Matrix:\n")
print(pivot)

plt.figure(figsize=(8,4))
plt.imshow(pivot,cmap="YlGnBu",aspect="auto")
plt.xticks(range(len(pivot.columns)),pivot.columns,rotation=45)
plt.yticks(range(len(pivot.index)),pivot.index)
plt.title("Person vs Category Expense Heatmap")
plt.colorbar(label="Amount")
plt.show()

# ===============================================================
# 13. AI Generated Insights
# ===============================================================
top_spender = person_total.idxmax()
least_spender = person_total.idxmin()

print("\nExpense Insights")
print("- Highest spender :", top_spender)
print("- Lowest spender  :", least_spender)
print("- Average Bill Value :", round(df['amount'].mean(),2))
print("- Median Bill Value  :", round(df['amount'].median(),2))
print("- Most Expensive Category :", cat_total.idxmax())
print("- Cheapest Category :", cat_total.idxmin())

# ===============================================================
# 14. Final Pipeline
# ===============================================================
print("\nPipeline:")
print("Receipt Image -> OCR -> Text -> Feature Extraction")
print("-> AI Classification -> Expense Analytics -> Forecasting")

# ReceiptDNA

In [ ]:
# ===============================================================
# ReceiptDNA - Visual Intelligence Edition
# Enterprise-Grade Human Expense Analytics
# ===============================================================

# ---------------------------------------------------------------
# 1. LIBRARY IMPORTS
# ---------------------------------------------------------------
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path
from datetime import datetime, timedelta
from scipy import stats
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, IsolationForest
from sklearn.metrics import silhouette_score, mean_absolute_error
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from statsmodels.tsa.holtwinters import ExponentialSmoothing
import networkx as nx

warnings.filterwarnings('ignore')

plt.style.use("seaborn-v0-8-darkgrid")
sns.set_palette("husl")
np.random.seed(42)

print("="*80)
print("ReceiptDNA - Visual Intelligence Edition")
print("Enterprise Expense Analytics Platform")
print("="*80)

# ---------------------------------------------------------------
# 2. DATA LOADING
# ---------------------------------------------------------------
DATASET = Path("/kaggle/input/datasets/urbikn/sroie-datasetv2/SROIE2019")
BOX = DATASET / "train" / "box"

files = list(BOX.glob("*.txt"))
SAMPLE_SIZE = min(200, len(files)) if len(files) > 200 else len(files)
files = files[:SAMPLE_SIZE]

# ---------------------------------------------------------------
# 3. PERSON PROFILES
# ---------------------------------------------------------------
people = [
    "Aarav", "Vivaan", "Aditya", "Rohan", "Kabir", "Ishaan", "Arjun", "Kunal",
    "Sneha", "Priya", "Riya", "Ananya", "Neha", "Pooja", "Diya", "Aditi",
    "Rahul", "Varun", "Meera", "Nidhi", "Tanya", "Sanya", "Yash", "Dev"
]

occupations = ["Engineer", "Doctor", "Teacher", "Student", "Entrepreneur", "Consultant"]
cities = ["Mumbai", "Delhi", "Bangalore", "Chennai", "Hyderabad", "Pune"]
income_groups = ["Low", "Medium-Low", "Medium", "Medium-High", "High"]

person_demographics = {
    name: {
        "occupation": np.random.choice(occupations),
        "city": np.random.choice(cities),
        "income_group": np.random.choice(income_groups, p=[0.1, 0.2, 0.4, 0.2, 0.1]),
        "age": np.random.randint(22, 60)
    }
    for name in people
}

# ---------------------------------------------------------------
# 4. RECEIPT PARSING ENGINE
# ---------------------------------------------------------------
def parse_receipt(file_path):
    """Extract structured data from receipt text"""
    try:
        text = open(file_path, encoding='utf-8').read().lower()
        
        number_patterns = re.findall(r'(\d+(?:\.\d{1,2})?)\s*(rs|inr|₹|$)?', text)
        amounts = [float(num) for num, _ in number_patterns if float(num) < 100000 and float(num) > 0]
        
        amount = max(amounts) if amounts else np.random.uniform(100, 5000)
        
        category_keywords = {
            "Food": ["pizza", "burger", "coffee", "tea", "restaurant", "cafe"],
            "Transport": ["uber", "taxi", "ride", "bus", "metro", "fuel"],
            "Healthcare": ["medicine", "tablet", "clinic", "hospital", "doctor"],
            "Groceries": ["rice", "milk", "fruit", "vegetable", "grocery"],
            "Shopping": ["mall", "retail", "cloth", "shoe", "electronic"],
            "Entertainment": ["movie", "cinema", "netflix", "spotify", "concert"],
            "Utilities": ["electricity", "water", "gas", "internet", "wifi"]
        }
        
        category_scores = {}
        for cat, keywords in category_keywords.items():
            score = sum(keyword in text for keyword in keywords)
            if score > 0:
                category_scores[cat] = score
        
        hour_pattern = re.search(r'(\d{1,2})[:](\d{2})', text)
        hour = int(hour_pattern.group(1)) if hour_pattern else np.random.randint(0, 24)
        
        time_of_day = (
            "Morning" if 5 <= hour < 12 else
            "Afternoon" if 12 <= hour < 17 else
            "Evening" if 17 <= hour < 21 else "Night"
        )
        
        return {
            "amount": amount,
            "category": max(category_scores, key=category_scores.get) if category_scores else "Other",
            "time_of_day": time_of_day,
            "confidence": len(amounts) / max(len(number_patterns), 1)
        }
    except Exception:
        return None

records = []
for file in files:
    result = parse_receipt(file)
    if result:
        result["receipt_id"] = file.stem
        records.append(result)

df = pd.DataFrame(records)
df["person"] = np.random.choice(people, size=len(df))
df["date"] = pd.date_range(start='2024-01-01', periods=len(df), freq='D')
df["day_of_week"] = df["date"].dt.day_name()
df["month"] = df["date"].dt.month

print(f"\nData Loading Summary")
print(f"Receipts loaded: {len(df)}")
print(f"Active persons: {df['person'].nunique()}")
print(f"Categories: {df['category'].nunique()}")

# ---------------------------------------------------------------
# 5. PERSON SUMMARIES
# ---------------------------------------------------------------
person_df = df.groupby("person").agg(
    total_spend=("amount", "sum"),
    avg_bill=("amount", "mean"),
    median_bill=("amount", "median"),
    std_bill=("amount", "std"),
    transactions=("receipt_id", "count"),
    unique_categories=("category", "nunique"),
    most_frequent_category=("category", lambda x: x.mode()[0] if len(x) > 0 else "None")
).reset_index()

for col in ["occupation", "city", "income_group", "age"]:
    person_df[col] = person_df["person"].map(lambda x: person_demographics[x][col])

person_df["spend_per_transaction"] = person_df["total_spend"] / person_df["transactions"]
person_df["category_diversity"] = person_df["unique_categories"] / df["category"].nunique()
person_df["spend_variability"] = person_df["std_bill"] / person_df["avg_bill"]

# ---------------------------------------------------------------
# 6. VISUALIZATION SUITE
# ---------------------------------------------------------------

# 6.1 Spending Distribution by Category
fig = px.box(df, x='category', y='amount', color='category',
             title='Spending Distribution by Category',
             labels={'amount': 'Amount (Currency Units)', 'category': 'Expense Category'})
fig.show()

# 6.2 Time Series Heatmap
pivot_time = df.pivot_table(values='amount', index='person', columns='day_of_week', 
                            aggfunc='mean', fill_value=0)
fig = px.imshow(pivot_time, text_auto=True, aspect="auto",
                title='Average Spend by Person and Day of Week',
                labels=dict(x="Day of Week", y="Person", color="Avg Spend"))
fig.show()

# 6.3 Category Spending Treemap
category_totals = df.groupby('category')['amount'].sum().reset_index()
fig = px.treemap(category_totals, path=['category'], values='amount',
                 title='Hierarchical Spending by Category',
                 color='amount', color_continuous_scale='Viridis')
fig.show()

# ---------------------------------------------------------------
# 7. CLUSTERING ANALYSIS
# ---------------------------------------------------------------
def perform_clustering(data):
    features = ['total_spend', 'avg_bill', 'transactions', 'spend_variability', 'category_diversity']
    X = data[features]
    
    imputer = SimpleImputer(strategy='median')
    X_imputed = imputer.fit_transform(X)
    
    scaler = RobustScaler()
    X_scaled = scaler.fit_transform(X_imputed)
    
    pca = PCA(n_components=2)
    X_pca = pca.fit_transform(X_scaled)
    
    silhouette_scores = []
    n_clusters_range = range(2, 7)
    
    for n in n_clusters_range:
        kmeans = KMeans(n_clusters=n, random_state=42, n_init=10)
        labels = kmeans.fit_predict(X_scaled)
        score = silhouette_score(X_scaled, labels)
        silhouette_scores.append(score)
    
    optimal_n = n_clusters_range[np.argmax(silhouette_scores)]
    kmeans = KMeans(n_clusters=optimal_n, random_state=42, n_init=10)
    data['cluster'] = kmeans.fit_predict(X_scaled)
    
    cluster_profiles = {}
    for cluster in data['cluster'].unique():
        cluster_data = data[data['cluster'] == cluster]
        avg_spend = cluster_data['total_spend'].mean()
        avg_freq = cluster_data['transactions'].mean()
        
        if avg_spend > data['total_spend'].mean() and avg_freq > data['transactions'].mean():
            profile = "High Spender"
        elif avg_spend > data['total_spend'].mean() and avg_freq <= data['transactions'].mean():
            profile = "Selective Premium"
        elif avg_spend <= data['total_spend'].mean() and avg_freq > data['transactions'].mean():
            profile = "Frequent Moderate"
        else:
            profile = "Conscious Saver"
        
        cluster_profiles[cluster] = profile
    
    data['financial_identity'] = data['cluster'].map(cluster_profiles)
    
    print(f"\nClustering Results")
    print(f"Optimal clusters: {optimal_n}")
    print(f"Silhouette score: {silhouette_scores[np.argmax(silhouette_scores)]:.3f}")
    
    return data, X_pca

person_df, pca_coords = perform_clustering(person_df)

# 6.4 Cluster Visualization
fig = px.scatter(person_df, x=pca_coords[:, 0], y=pca_coords[:, 1],
                 color='financial_identity', size='total_spend',
                 hover_name='person', text='person',
                 title='Financial Identity Clustering (PCA Projection)',
                 labels={'x': 'Principal Component 1', 'y': 'Principal Component 2'})
fig.show()

# ---------------------------------------------------------------
# 8. TIME SERIES FORECASTING
# ---------------------------------------------------------------
def forecast_spending(df_transactions):
    daily_spend = df_transactions.groupby('date')['amount'].sum().reset_index()
    daily_spend.set_index('date', inplace=True)
    
    full_range = pd.date_range(start=daily_spend.index.min(), end=daily_spend.index.max(), freq='D')
    daily_spend = daily_spend.reindex(full_range, fill_value=0)
    
    X = np.arange(len(daily_spend)).reshape(-1, 1)
    y = daily_spend['amount'].values
    
    train_size = int(len(y) * 0.8)
    X_train, X_test = X[:train_size], X[train_size:]
    y_train, y_test = y[:train_size], y[train_size:]
    
    models = {
        'Linear Regression': LinearRegression(),
        'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
        'Ridge': Ridge(alpha=1.0)
    }
    
    predictions = {}
    for name, model in models.items():
        model.fit(X_train, y_train)
        predictions[name] = model.predict(X_test)
    
    ensemble_pred = np.mean(list(predictions.values()), axis=0)
    mae = mean_absolute_error(y_test, ensemble_pred)
    
    future_X = np.arange(len(y), len(y) + 7).reshape(-1, 1)
    future_pred = np.mean([model.predict(future_X) for model in models.values()], axis=0)
    
    last_date = daily_spend.index[-1]
    future_dates = [last_date + timedelta(days=i+1) for i in range(7)]
    
    forecast_df = pd.DataFrame({
        'date': future_dates,
        'predicted_amount': future_pred,
        'lower_bound': future_pred * 0.8,
        'upper_bound': future_pred * 1.2
    })
    
    return forecast_df, mae, daily_spend

forecast_7days, forecast_accuracy, daily_trend = forecast_spending(df)

# 6.5 Forecast Visualization
fig = make_subplots(rows=2, cols=1, 
                    subplot_titles=('Daily Spend with 7-Day Forecast', 'Weekly Spending Pattern'),
                    vertical_spacing=0.15)

fig.add_trace(go.Scatter(x=daily_trend.index, y=daily_trend['amount'],
                         mode='lines+markers', name='Historical',
                         line=dict(color='blue', width=2)),
              row=1, col=1)

fig.add_trace(go.Scatter(x=forecast_7days['date'], y=forecast_7days['predicted_amount'],
                         mode='lines+markers', name='Forecast',
                         line=dict(color='red', width=2, dash='dash')),
              row=1, col=1)

fig.add_trace(go.Scatter(x=forecast_7days['date'], y=forecast_7days['upper_bound'],
                         fill=None, mode='lines', line=dict(width=0), showlegend=False),
              row=1, col=1)

fig.add_trace(go.Scatter(x=forecast_7days['date'], y=forecast_7days['lower_bound'],
                         fill='tonexty', mode='lines', line=dict(width=0),
                         name='Confidence Interval (80%)', fillcolor='rgba(255,0,0,0.2)'),
              row=1, col=1)

weekly_pattern = df.groupby('day_of_week')['amount'].mean().reindex(
    ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
)

fig.add_trace(go.Bar(x=weekly_pattern.index, y=weekly_pattern.values,
                     name='Average Daily Spend', marker_color='lightgreen'),
              row=2, col=1)

fig.update_layout(height=800, title_text="Time Series Analysis", showlegend=True)
fig.show()

print(f"\nForecasting Results")
print(f"Forecast accuracy (MAE): {forecast_accuracy:.2f}")
print(f"Next 7 days total spend: {forecast_7days['predicted_amount'].sum():.2f}")

# ---------------------------------------------------------------
# 9. ANOMALY DETECTION
# ---------------------------------------------------------------
def detect_anomalies(data):
    features = ['total_spend', 'avg_bill', 'transactions']
    X = data[features]
    
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    iso_forest = IsolationForest(contamination=0.1, random_state=42)
    anomalies = iso_forest.fit_predict(X_scaled)
    
    data['is_anomaly'] = anomalies == -1
    data['anomaly_score'] = iso_forest.score_samples(X_scaled)
    
    return data

person_df = detect_anomalies(person_df)

fig = px.scatter(person_df, x='transactions', y='total_spend',
                 color='is_anomaly', size='anomaly_score',
                 hover_name='person', text='person',
                 title='Spending Anomaly Detection',
                 labels={'transactions': 'Transaction Count', 'total_spend': 'Total Spend'},
                 color_discrete_map={True: 'red', False: 'blue'})
fig.show()

# ---------------------------------------------------------------
# 10. CORRELATION ANALYSIS
# ---------------------------------------------------------------
numeric_cols = ['total_spend', 'avg_bill', 'transactions', 'age', 'spend_variability']
correlation_matrix = person_df[numeric_cols].corr()

fig = px.imshow(correlation_matrix, text_auto=True, aspect="auto",
                title='Correlation Matrix: Demographics and Spending Behavior',
                color_continuous_scale='RdBu', zmin=-1, zmax=1)
fig.show()

# ---------------------------------------------------------------
# 11. INSIGHTS GENERATION
# ---------------------------------------------------------------
print("\n" + "="*80)
print("ANALYTICS DASHBOARD")
print("="*80)

print(f"\nSpending Performance")
print(f"  Highest Spender: {person_df.loc[person_df['total_spend'].idxmax(), 'person']}")
print(f"  Most Frequent: {person_df.loc[person_df['transactions'].idxmax(), 'person']}")
print(f"  Most Diverse: {person_df.loc[person_df['category_diversity'].idxmax(), 'person']}")

print(f"\nCategory Analysis")
category_stats = df.groupby('category')['amount'].agg(['mean', 'sum']).round(2)
for category in category_stats.index:
    print(f"  {category}: Avg {category_stats.loc[category, 'mean']} | Total {category_stats.loc[category, 'sum']}")

print(f"\nTime Patterns")
time_patterns = df.groupby('time_of_day')['amount'].mean().round(2)
for time_of_day, avg in time_patterns.items():
    print(f"  {time_of_day}: {avg} average spend")

print(f"\nAnomaly Detection")
print(f"  Anomalies detected: {person_df['is_anomaly'].sum()} persons")
if person_df['is_anomaly'].sum() > 0:
    anomalies = person_df[person_df['is_anomaly']][['person', 'total_spend', 'avg_bill', 'transactions']]
    print(anomalies.to_string(index=False))

print(f"\nGeographic Analysis")
city_ranking = person_df.groupby('city')['total_spend'].mean().sort_values(ascending=False)
for city, avg_spend in city_ranking.head(3).items():
    print(f"  {city}: {avg_spend:.2f} average spend")

# ---------------------------------------------------------------
# 12. PREDICTIVE RECOMMENDATIONS
# ---------------------------------------------------------------
print(f"\nPredictive Recommendations")

for _, person in person_df.head(5).iterrows():
    recommendations = []
    
    if person['spend_variability'] > 1.0:
        recommendations.append("High spending volatility detected - consider budget planning")
    
    if person['category_diversity'] < 0.3:
        recommendations.append("Low category diversity - explore new spending areas")
    
    if person['transactions'] > person_df['transactions'].median() and person['avg_bill'] < person_df['avg_bill'].median():
        recommendations.append("Frequent small transactions - consider subscription consolidation")
    
    if person['is_anomaly']:
        recommendations.append("Unusual spending pattern detected - review recent transactions")
    
    if recommendations:
        print(f"\n  {person['person']} ({person['financial_identity']}):")
        for rec in recommendations:
            print(f"    - {rec}")

# ---------------------------------------------------------------
# 13. EXECUTIVE SUMMARY
# ---------------------------------------------------------------
print("\n" + "="*80)
print("EXECUTIVE SUMMARY")
print("="*80)
print(f"Total Receipts Processed: {len(df)}")
print(f"Total Persons Analyzed: {len(person_df)}")
print(f"Total Spend Analyzed: {df['amount'].sum():,.2f}")
print(f"Average Transaction Value: {df['amount'].mean():.2f}")
print(f"Predicted Next Week Spend: {forecast_7days['predicted_amount'].sum():.2f}")
print(f"Primary Spending Category: {df['category'].mode()[0]}")
print("="*80)

# Optional: Export results
# person_df.to_csv('receiptdna_person_insights.csv', index=False)
# df.to_csv('receiptdna_transactions.csv', index=False)
# forecast_7days.to_csv('receiptdna_forecast.csv', index=False)